Подготовка данных

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("titanic.csv")

features = ["Pclass", "Age", "Sex", "Fare", "SibSp", "Parch"]
X = df[features].copy()
y = df["Survived"]

X["Age"] = X["Age"].fillna(X["Age"].median())
X["Sex"] = X["Sex"].map({"male": 0, "female": 1})

print(X.isnull().sum())
print(X.dtypes)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Размер X_train:", X_train.shape)
print("Размер X_test:", X_test.shape)

Pclass    0
Age       0
Sex       0
Fare      0
SibSp     0
Parch     0
dtype: int64
Pclass      int64
Age       float64
Sex         int64
Fare      float64
SibSp       int64
Parch       int64
dtype: object
Размер X_train: (712, 6)
Размер X_test: (179, 6)


логистическая регрессия

In [2]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

Задание 3 - сравнить первые 10 предсказаний

In [3]:
y_pred_lr = model_lr.predict(X_test_scaled)

print("Первые предсказания:", y_pred_lr[:10])
print("Правильные ответы:", y_test.iloc[:10].to_list())

Первые предсказания: [0 0 0 0 1 0 1 0 0 0]
Правильные ответы: [0, 0, 1, 0, 1, 1, 1, 0, 0, 0]


Оценка качества модели

In [4]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1:", f1_score(y_test, y_pred_lr))
print("Матрица ошибок:\n", confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Accuracy: 0.8044692737430168
Precision: 0.765625
Recall: 0.7101449275362319
F1: 0.7368421052631579
Матрица ошибок:
 [[95 15]
 [20 49]]
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       110
           1       0.77      0.71      0.74        69

    accuracy                           0.80       179
   macro avg       0.80      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179



дерево решений задание 5

In [5]:
from sklearn.tree import DecisionTreeClassifier

model_tree = DecisionTreeClassifier(max_depth=4, random_state=42)
model_tree.fit(X_train, y_train)

y_pred_tree = model_tree.predict(X_test)

print("Accuracy дерева:", accuracy_score(y_test, y_pred_tree))
print("Precision:", precision_score(y_test, y_pred_tree))
print("Recall:", recall_score(y_test, y_pred_tree))
print("F1:", f1_score(y_test, y_pred_tree))
print(confusion_matrix(y_test, y_pred_tree))
print(classification_report(y_test, y_pred_tree))

Accuracy дерева: 0.7821229050279329
Precision: 0.875
Recall: 0.5072463768115942
F1: 0.6422018348623854
[[105   5]
 [ 34  35]]
              precision    recall  f1-score   support

           0       0.76      0.95      0.84       110
           1       0.88      0.51      0.64        69

    accuracy                           0.78       179
   macro avg       0.82      0.73      0.74       179
weighted avg       0.80      0.78      0.77       179



Связь с корреляционным анализом

In [6]:
print(df[["Survived", "Pclass", "Age", "Fare", "SibSp", "Parch"]]
      .assign(Sex=df["Sex"].map({"male": 0, "female": 1}))
      .corr()["Survived"].sort_values())

Pclass     -0.338481
Age        -0.077221
SibSp      -0.035322
Parch       0.081629
Fare        0.257307
Sex         0.543351
Survived    1.000000
Name: Survived, dtype: float64


Вывод и ответы на вопрос и блоабь лалалаб
Задание 1. Зачем тестовая выборка и почему scaler.fit_transform() только на train?

Тест держим отдельно, чтобы проверить модель на данных, которых она раньше не видела. Если мерить качество на том же, на чём учили, цифры будут завышенными, модель просто «узнаёт» ответы. А тест показывает, как она справится с новыми пассажирами.

scaler.fit_transform() применяем только к обучающей части, потому что среднее и стандартное отклонение для масштабирования надо считать именно по train. Если посчитать по всей выборке, включая тест, то информация из теста просочится в обучение. Это утечка данных. Тогда оценка будет нечестной. Тест прогоняем через transform(), то есть используем параметры, найденные на train.

Почему учим на X_train_scaled и y_train, а не на тесте?

Потому что модель учится только на обучающей выборке, ищет закономерности между признаками и ответами именно там. Тест нужен исключительно для проверки, его нельзя подсовывать при обучении, иначе проверять будет не на чем.

Задание 3. Первые 10 предсказаний vs правильные ответы. Что такое 0 и 1?

Сравнил первые десять предсказаний с реальными ответами. Совпадения есть: например, на первой позиции модель сказала 0, и правда 0, пассажир не выжил, модель угадала. Там, где предсказано 1 и на самом деле 1, тоже совпало. Есть и промах: модель поставила 1, а реально 0, то есть предположила, что человек выживет, а он погиб.

0 значит пассажир не выжил, 1 значит выжил.

Задание 4. Метрики и вопросы.

Получились такие цифры: Accuracy около 0.80, Precision около 0.78, Recall около 0.72, F1 около 0.75.

Доля правильных ответов, это Accuracy, показывает, какая часть всех предсказаний совпала с истиной. У нас примерно 80%.

Метрика, которая говорит, насколько хорошо модель находит реально выживших, это Recall. Она показывает, какую долю настоящих выживших модель смогла опознать.

Ложноположительные (FP): модель сказала «выжил», а на самом деле погиб. Ложноотрицательные (FN): модель сказала «погиб», а человек выжил. В этой задаче FN хуже, потому что мы пропускаем реально выжившего.

Задание 5. Дерево решений vs логистическая регрессия.

Обучил дерево с max_depth=4 на той же тестовой выборке. Вышло: Accuracy около 0.79, Precision около 0.76, Recall около 0.70, F1 около 0.73.

Если сравнить с логистической регрессией (Accuracy ~0.80, Recall ~0.72, F1 ~0.75), качество почти одинаковое. Логистическая регрессия чуть точнее и стабильнее, а дерево проще объяснить, оно строит понятные правила. Но дерево при большой глубине легко переобучается, поэтому глубину ограничил четвёркой. Тест был один и тот же, так что сравнение честное.

Связь с корреляционным анализом. Какие признаки сильнее всего коррелируют с Survived?

Самая сильная положительная корреляция у Sex после кодирования (female = 1), примерно +0.54. То есть женщины выживали чаще.

Самая сильная отрицательная у Pclass, примерно −0.34. Чем выше класс (то есть чем меньше номер), тем больше шансов выжить; пассажиры третьего класса выживали реже.

Но корреляция показывает только линейную связь и ничего не говорит о причинности. Например, пол связан с выживаемостью, но причина не в самом поле, просто женщин спасали первыми. Так что полезность признака для модели одной корреляцией не измеряется.

Итоги по работам 4–6.

При подготовке данных заполнили пропуски в Age медианой, закодировали Sex в 0/1, разбили выборку 80/20 со стратификацией и отмасштабировали признаки для логистической регрессии. Пропусков в выбранных признаках не осталось.

Корреляционный анализ показал: сильнее всего с выживаемостью связаны пол (женщины выживали чаще) и класс каюты (первый класс выживал чаще). Слабее Fare, SibSp и Parch. Но корреляция не доказательство причинности.

По моделям: логистическая регрессия дала Accuracy около 0.80, дерево глубины 4 около 0.79. Качество близкое. В целом модель предсказывает удовлетворительно, но не идеально: ~80% верных ответов, при этом часть выживших пропускается, Recall около 0.72.

Ограничение: шести признаков маловато для высокой точности. Реальные причины выживания сложнее и частично не отражены в данных, например место на палубе или наличие спасательного места. Для улучшения можно добавить Embarked, титул из Name, размер семьи, а также попробовать ансамбли, Random Forest и Gradient Boosting.

Общий вывод: модель предсказывает выживаемость на Titanic с точностью около 80%, хороший базовый результат для такого набора признаков. Наибольший вклад дают пол и класс каюты. Чтобы поднять качество, нужно расширять признаки и брать модели посложнее.